# LSTM vs SimpleRNN on a long-memory task
Real IMDB reviews: we keep the first 50 words of each review and then add a gap of padding steps after it.
The sentiment is in the words, so the model must carry it across the gap to the last time step.
Same data, same sizes, same training; only the recurrent layer (SimpleRNN or LSTM) and the gap change.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

try:
    tf.config.experimental.enable_op_determinism()   # same numbers on every run
except Exception as e:
    print("op determinism not available:", e)
print("GPUs:", tf.config.list_physical_devices("GPU"))
DATA = Path("data")
DATA.mkdir(exist_ok=True)

## The data
5,000 training and 5,000 test reviews, 10,000 most frequent words. Each review is cut to its first 50 words
(shorter ones are padded in front), then `gap` padding steps (index 0) follow.

In [ ]:
VOCAB, KEEP, N = 10000, 50, 5000
(x_tr, y_tr), (x_te, y_te) = keras.datasets.imdb.load_data(num_words=VOCAB)
x_tr, x_te = x_tr[:N], x_te[:N]
y_tr, y_te = y_tr[:N].astype("float32"), y_te[:N].astype("float32")


def with_gap(x, gap):
    first = keras.utils.pad_sequences(x, maxlen=KEEP, padding="pre", truncating="post")   # first 50 words
    return np.concatenate([first, np.zeros((len(x), gap), dtype="int32")], axis=1).astype("int32")


print("shape with gap 25:", with_gap(x_tr, 25).shape, " positive share:", y_tr.mean())

## Training
Embedding (32 numbers per word), a recurrent layer of 32 units, a sigmoid output. Adam, 8 epochs, batch 64.
Every setting is trained with 3 seeds.

In [ ]:
GAPS, SEEDS, EPOCHS = (0, 25), (0, 1, 2), 8
rows = []
for gap in GAPS:
    xa, xb = with_gap(x_tr, gap), with_gap(x_te, gap)
    for kind in ("SimpleRNN", "LSTM"):
        for seed in SEEDS:
            keras.utils.set_random_seed(seed)
            model = keras.Sequential([keras.Input(shape=(KEEP + gap,)),
                                      keras.layers.Embedding(VOCAB, 32),
                                      getattr(keras.layers, kind)(32),
                                      keras.layers.Dense(1, activation="sigmoid")])
            model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
            h = model.fit(xa, y_tr, epochs=EPOCHS, batch_size=64, validation_data=(xb, y_te), verbose=0)
            for e, acc in enumerate(h.history["val_accuracy"], start=1):
                rows.append(dict(gap=gap, model=kind, seed=seed, epoch=e, val_accuracy=acc))
            print(f"gap {gap:3d} {kind:9} seed {seed}: final {h.history['val_accuracy'][-1]:.3f}", flush=True)
curves = pd.DataFrame(rows)
curves.round(4).to_csv(DATA / "val_accuracy.csv", index=False)

## Results: mean over 3 seeds

In [ ]:
final = curves[curves.epoch == EPOCHS].groupby(["gap", "model"]).val_accuracy.agg(["mean", "min", "max"]).round(3)
print(final)
print("chance level (share of the larger class in the test set):", round(max(y_te.mean(), 1 - y_te.mean()), 3))